## Leave one session out crsso validation
### MDM RIEMANN

In [4]:
from pyriemann.estimation import Covariances
from pyriemann.classification import MDM
from sklearn.metrics import (balanced_accuracy_score,accuracy_score,
                            confusion_matrix, classification_report)

def riemannian_mdm(X, y, subject_ids, session_ids):
    """
    Minimum Distance to Mean classifier.

    """
    results = []

    for subj in np.unique(subject_ids):
        mask        = subject_ids == subj
        X_s         = X[mask]
        y_s         = y[mask]
        sess_s      = session_ids[mask]
        unique_sess = np.unique(sess_s)

        if len(np.unique(y_s)) < 2:
            continue

        session_scores = []

        for test_sess in unique_sess: #each session becomes test
            train_mask = sess_s != test_sess
            test_mask  = sess_s == test_sess

            X_train = X_s[train_mask]
            X_test  = X_s[test_mask]
            y_train = y_s[train_mask]
            y_test  = y_s[test_mask]

            if len(np.unique(y_train)) < 2 or \
               len(np.unique(y_test))  < 2:
                continue

            try:
                #  covariance matrices
                cov_est   = Covariances(estimator='oas')
                X_tr_cov  = cov_est.fit_transform(X_train)
                X_te_cov  = cov_est.transform(X_test)

                # MDM classifier 
                mdm = MDM(metric='riemann')
                mdm.fit(X_tr_cov, y_train)
                preds = mdm.predict(X_te_cov)
                probs = mdm.predict_proba(X_te_cov)[:, 1]
                acc = balanced_accuracy_score(y_test, preds)
                f1 = f1_score(y_test, preds, average='macro')
                try:
                    auc = roc_auc_score(y_test, probs)
                except ValueError:
                    auc = np.nan
                session_scores.append({
                    'acc': acc,
                    'f1': f1,
                    'auc': auc
                })
                
                print(f"S{subj} Sess {test_sess} -> Acc: {acc:.3f} | F1: {f1:.3f} | AUC: {auc:.3f}")
            

            except Exception as e:
                print(e)

        if session_scores:
           
            subj_df = pd.DataFrame(session_scores)
            
  
            results.append({
                'subject': subj,
                'mean_acc': subj_df['acc'].mean(),
                'std_acc':  subj_df['acc'].std(),
                'mean_f1':  subj_df['f1'].mean(),
                'std_f1':   subj_df['f1'].std(),
                'mean_auc': subj_df['auc'].mean(),
                'std_auc':  subj_df['auc'].std()
            })
    # ==========================================
    # FINAL DATAFRAME & GRAND MEANS
    # ==========================================
    final_df = pd.DataFrame(results)

    print("\n=======================================================================")
    print("OVERALL METRICS PER SUBJECT")
    print("=======================================================================")
    print(final_df.set_index('subject').round(3))

    print("\n=======================================================================")
    print("GRAND MEANS (Συνολική Απόδοση Συστήματος)")
    print("=======================================================================")
    print(f"Balanced Accuracy: {final_df['mean_acc'].mean():.3f} ± {final_df['mean_acc'].std():.3f}")
    print(f"F1-Score (Macro):  {final_df['mean_f1'].mean():.3f} ± {final_df['mean_f1'].std():.3f}")
    print(f"ROC-AUC:           {final_df['mean_auc'].mean():.3f} ± {final_df['mean_auc'].std():.3f}")
    print("=======================================================================")
    
    return final_df


### Multiwindow attempt with Tangent Space and Logistic Regression

In [ ]:
import numpy as np
import pandas as pd

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score,f1_score, roc_auc_score
from sklearn.decomposition import PCA

def multiwindow_riemann(
    X,
    y,
    subject_ids,
    session_ids,
    sfreq=128
):

    results = []

    # windows in seconds
    windows_sec = [
        (-3.0, -2.5),
        (-2.5, -2.0),
        (-2.0, -1.5),
        (-1.5, -1.0),
        (-1.0, -0.5),
        (-0.5, 0.0),
    ]

    n_times = X.shape[2]

    for subj in np.unique(subject_ids):

        subj_mask = subject_ids == subj

        X_s = X[subj_mask]
        y_s = y[subj_mask]
        sess_s = session_ids[subj_mask]

        session_scores = []

        for test_sess in np.unique(sess_s):

            train_mask = sess_s != test_sess
            test_mask  = sess_s == test_sess

            X_train = X_s[train_mask]
            y_train = y_s[train_mask]

            X_test  = X_s[test_mask]
            y_test  = y_s[test_mask]

            if len(np.unique(y_train)) < 2:
                continue

            try:

                train_features = []
                test_features  = []

                # ==========================================
                # MULTI-WINDOW FEATURE EXTRACTION
                # ==========================================

                for t0, t1 in windows_sec:

                    s0 = int((t0 + 3.0) * sfreq) # each second translates to a specific sample
                    s1 = int((t1 + 3.0) * sfreq)

                    Xtr_w = X_train[:, :, s0:s1]
                    Xte_w = X_test[:, :, s0:s1]

                    # covariance
                    cov = Covariances(estimator='oas')

                    Xtr_cov = cov.fit_transform(Xtr_w)
                    Xte_cov = cov.transform(Xte_w)

                    # tangent space
                    ts = TangentSpace()

                    Xtr_ts = ts.fit_transform(Xtr_cov)
                    Xte_ts = ts.transform(Xte_cov)

                    train_features.append(Xtr_ts)
                    test_features.append(Xte_ts)

                # concatenate all windows
                Xtr_final = np.concatenate(train_features, axis=1)
                Xte_final = np.concatenate(test_features, axis=1)
               # pca = PCA(
                #    n_components=0.95
                #)

                #Xtr_final = pca.fit_transform(Xtr_final)
                #Xte_final = pca.transform(Xte_final)

                # ==========================================
                # CLASSIFIER
                # ==========================================

                clf = LogisticRegression(
                    max_iter=4000,
                    class_weight='balanced',
                    C=0.1,
                    solver='liblinear'
                )

                clf.fit(Xtr_final, y_train)

                preds = clf.predict(Xte_final)
                probs = clf.predict_proba(Xte_final)[:, 1]

                acc = balanced_accuracy_score(y_test, preds)
                f1 = f1_score(y_test, preds, average='macro')
                try:
                    auc = roc_auc_score(y_test, probs)
                except ValueError:
                    auc = np.nan
                session_scores.append({
                    'acc': acc,
                    'f1': f1,
                    'auc': auc
                })
                
                print(
                    print(f"S{subj} Sess {test_sess} -> Acc: {acc:.3f} | F1: {f1:.3f} | AUC: {auc:.3f}")
                )

            except Exception as e:
                print(e)

        if session_scores:
           
            subj_df = pd.DataFrame(session_scores)
            
            results.append({
                'subject': subj,
                'mean_acc': subj_df['acc'].mean(),
                'std_acc':  subj_df['acc'].std(),
                'mean_f1':  subj_df['f1'].mean(),
                'std_f1':   subj_df['f1'].std(),
                'mean_auc': subj_df['auc'].mean(),
                'std_auc':  subj_df['auc'].std()
            })

    # ==========================================
    # FINAL DATAFRAME & GRAND MEANS
    # ==========================================
    final_df = pd.DataFrame(results)

    print("\n=======================================================================")
    print("OVERALL METRICS PER SUBJECT")
    print("=======================================================================")
    print(final_df.set_index('subject').round(3))

    print("\n=======================================================================")
    print("GRAND MEANS (Συνολική Απόδοση Συστήματος)")
    print("=======================================================================")
    print(f"Balanced Accuracy: {final_df['mean_acc'].mean():.3f} ± {final_df['mean_acc'].std():.3f}")
    print(f"F1-Score (Macro):  {final_df['mean_f1'].mean():.3f} ± {final_df['mean_f1'].std():.3f}")
    print(f"ROC-AUC:           {final_df['mean_auc'].mean():.3f} ± {final_df['mean_auc'].std():.3f}")
    print("=======================================================================")
    
    return final_df

### EEGNET with early stopping

In [ ]:
git clone https://github.com/vlawhern/arl-eegmodels.git


In [ ]:
cd arl-eegmodels

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
from EEGModels import EEGNet
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
from EEGModels import EEGNet
import random
import os
from sklearn.model_selection import train_test_split
SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

tf.config.experimental.enable_op_determinism()

def eegnet_cv(X, y, subject_ids, session_ids):

    results = []

    Chans = X.shape[1]
    Samples = X.shape[2]

    X_reshaped = X[..., np.newaxis]

    fold_results = []
    all_preds = np.full(len(y), -1)
    all_trues = np.full(len(y), -1)

    fold_counter = 0

    for subj in np.unique(subject_ids):

        mask = (subject_ids == subj)

        X_s = X_reshaped[mask]
        y_s = y[mask]
        sess_s = session_ids[mask]

        unique_sess = np.unique(sess_s)

        if len(np.unique(y_s)) < 2:
            print(f"Skipping S{subj}: Not enough classes.")
            continue

        session_accs = []

        for test_sess in unique_sess:

            train_mask = (sess_s != test_sess)
            test_mask = (sess_s == test_sess)
            
            X_train = X_s[train_mask]
            X_test = X_s[test_mask]
            
            y_train = y_s[train_mask]
            y_test = y_s[test_mask]
            X_tr, X_val, y_tr, y_val = train_test_split(
                    X_train,
                    y_train,
                    test_size=0.2,
                    stratify=y_train,
                    random_state=SEED
                    )
            if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                continue

            # z-score ormalization with training set
            train_mean = X_tr.mean(axis=(0, 2), keepdims=True)
            train_std = X_tr.std(axis=(0, 2), keepdims=True) + 1e-8

            X_train_norm = (X_tr - train_mean) / train_std
            X_test_norm = (X_test - train_mean) / train_std
            X_val_norm = (X_val - train_mean) / train_std
            # Class weights
            classes = np.unique(y_tr)

            weights = compute_class_weight(
                class_weight='balanced',
                classes=classes,
                y=y_tr
            )

            class_weight_dict = dict(zip(classes, weights))

            try:

                tf.keras.backend.clear_session()
                tf.random.set_seed(SEED)

                model = EEGNet(
                    nb_classes=2,
                    Chans=Chans,
                    Samples=Samples,
                    dropoutRate=0.5,
                    kernLength=64,
                    F1=8,
                    D=2,
                    F2=16,
                    dropoutType='Dropout'
                )

                model.compile(
                    loss='sparse_categorical_crossentropy',
                    optimizer='adam',
                    metrics=['accuracy']
                )
                model.summary()

                early_stop = EarlyStopping(
                    monitor='val_loss',
                    patience=10,
                    restore_best_weights=True
                )

                history= model.fit(
                        X_train_norm ,
                        y_tr,
                        batch_size=16,
                        epochs=50,
                        verbose=0,
                        validation_data=(X_val_norm, y_val),
                        callbacks=[early_stop],
                        class_weight=class_weight_dict
                )
                

                probs = model.predict(X_test_norm, verbose=0)
                preds = probs.argmax(axis=-1)

                bal_acc = balanced_accuracy_score(y_test, preds)
                acc = accuracy_score(y_test, preds)

                precision = precision_score(y_test, preds)
                recall = recall_score(y_test, preds)
                f1 = f1_score(y_test, preds)

                roc_auc = roc_auc_score(y_test, probs[:, 1])
                

                # confusion matrix
                cm_local = confusion_matrix(y_test, preds)
                tn, fp, fn, tp = cm_local.ravel()

                specificity = tn / (tn + fp + 1e-8)

                session_accs.append(bal_acc)

                fold_counter += 1

                print(
                    f"\n── Fold {fold_counter} | "
                    f"Subj {subj} | "
                    f"Leaving Out Sess {test_sess} ──"
                )

                print(f"Balanced Accuracy: {bal_acc:.3f}")

                global_idx = np.where(mask)[0][test_mask]

                all_preds[global_idx] = preds
                all_trues[global_idx] = y_test

                fold_results.append({
                    'fold': fold_counter,
                    'subject': subj,
                    'session': test_sess,
                    'accuracy': acc,
                    'bal_acc': bal_acc,
                    'precision': precision,
                    'recall': recall,
                    'specificity': specificity,
                    'f1': f1,
                    'roc_auc': roc_auc,
                    
                })
                print(f"Accuracy          : {acc:.3f}")
                print(f"Balanced Accuracy : {bal_acc:.3f}")
                print(f"Precision         : {precision:.3f}")
                print(f"Recall/Sensitivity: {recall:.3f}")
                print(f"Specificity       : {specificity:.3f}")
                print(f"F1 Score          : {f1:.3f}")
                print(f"ROC-AUC           : {roc_auc:.3f}")
            
            except Exception as e:

                print(f"Error in S{subj} session {test_sess}: {e}")
                continue
            

        if session_accs:
            subj_folds = [r for r in fold_results if r['subject'] == subj]

            mean_acc = np.mean([r['accuracy'] for r in subj_folds])
            std_acc = np.std([r['accuracy'] for r in subj_folds])
            mean_bal_acc = np.mean([r['bal_acc'] for r in subj_folds])
            std_bal_acc = np.std([r['bal_acc'] for r in subj_folds])
            mean_f1 = np.mean([r['f1'] for r in subj_folds])
            std_f1 = np.std([r['f1'] for r in subj_folds])
            mean_roc = np.mean([r['roc_auc'] for r in subj_folds])
            std_roc = np.std([r['roc_auc'] for r in subj_folds])
            print(f"\n=== Subject {subj} Summary ===")
            print(f"Mean Accuracy      : {mean_acc:.3f}")
            print(f"std Accuracy      : {std_acc:.3f}")
            print(f"Mean Balanced Acc  : {mean_bal_acc:.3f}")
            print(f"std Balanced Accuracy      : {std_bal_acc:.3f}")
            print(f"Mean F1      : {mean_f1:.3f}")
            print(f"std F1      : {std_f1:.3f}")
            print(f"Mean roc  : {mean_roc:.3f}")
            print(f"std roc      : {std_roc:.3f}")
            

            results.append({
                'subject': subj,
                'mean_acc': mean_acc,
                'std_acc': std_acc,
                'bal_ac': mean_bal_acc,
                'std_bal_Acc':std_bal_acc,
                'mean_f1' : mean_f1 ,
                'std_f1' :std_f1,
                'mean_roc' :mean_roc,
                'std_roc' : std_roc
            })

            

    

    # Final summary
    if fold_results:

        bal_accs = [r['bal_acc'] for r in fold_results]
        f1_scores = [r['f1'] for r in fold_results]
        auc_roc = [r['roc_auc'] for r in fold_results]
        print(f"\n{'='*65}")
        print(f"Total Valid Splits: {len(fold_results)}")
        print(f"Balanced Accuracy : {np.mean(bal_accs):.3f} ± {np.std(bal_accs):.3f}")
        print(f"F1 Score : {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
        print(f"AUC ROC : {np.mean(auc_roc):.3f} ± {np.std(auc_roc):.3f}")
        
        print(f"{'='*65}")

    # Confusion matrix
    valid_mask = (all_trues != -1) & (all_preds != -1)

    

    cm = confusion_matrix(
        all_trues[valid_mask],
        all_preds[valid_mask]
    )

    print(classification_report(
        all_trues[valid_mask],
        all_preds[valid_mask]
    ))

    print("\nConfusion Matrix:")
    print(cm)

    

    return results, fold_results, all_preds, all_trues



## Leave one subect out cross validation
#### MDM RIEMANN

In [ ]:
import numpy as np
import pandas as pd
from pyriemann.estimation import Covariances
from pyriemann.classification import MDM
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

def riemannian_mdm_cross(X, y, subject_ids, session_ids=None):
    """
    Minimum Distance to Mean classifier using Leave-One-Subject-Out (LOSO) CV.
    """
    results = []
    

    unique_subjects = np.unique(subject_ids) 
    
    for subj in unique_subjects: #ceach subject becomes test set
        train_mask = (subject_ids != subj)
        test_mask  = (subject_ids == subj)
        
        X_train = X[train_mask]
        X_test = X[test_mask]
        
        y_train = y[train_mask]
        y_test = y[test_mask]

        # Skip if there aren't at least 2 classes in either train or test sets
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(f"Skipping Subject {subj}: Not enough classes.")
            continue

        try:
            # covariance matrices
            cov_est   = Covariances(estimator='oas')
            X_tr_cov  = cov_est.fit_transform(X_train)
            X_te_cov  = cov_est.transform(X_test)

            # MDM classifier 
            mdm = MDM(metric='riemann')
            mdm.fit(X_tr_cov, y_train)
            
            preds = mdm.predict(X_te_cov)
            probs = mdm.predict_proba(X_te_cov)[:, 1]
            
            acc = balanced_accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds, average='macro')
            
            try:
                auc = roc_auc_score(y_test, probs)
            except ValueError:
                auc = np.nan
                
    
            results.append({
                'subject': subj,
                'acc': acc,
                'f1':  f1,
                'auc': auc
            })

        except Exception as e:
            print(f"Error processing subject {subj}: {e}")

    # ==========================================
    # FINAL DATAFRAME & GRAND MEANS
    # ==========================================
    final_df = pd.DataFrame(results)

    print("\n=======================================================================")
    print("OVERALL METRICS PER SUBJECT")
    print("=======================================================================")
    print(final_df.set_index('subject').round(3))

    print("\n=======================================================================")
    print("GRAND MEANS (Συνολική Απόδοση Συστήματος)")
    print("=======================================================================")
    
  
    print(f"Balanced Accuracy: {final_df['acc'].mean():.3f} ± {final_df['acc'].std():.3f}")
    print(f"F1-Score (Macro):  {final_df['f1'].mean():.3f} ± {final_df['f1'].std():.3f}")
    print(f"ROC-AUC:           {final_df['auc'].mean():.3f} ± {final_df['auc'].std():.3f}")
    print("=======================================================================")
    
    return final_df

### Multiwindow Riemann

In [ ]:
import numpy as np
import pandas as pd

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
# from sklearn.decomposition import PCA  

def multiwindow_riemann_cross(X, y, subject_ids, session_ids=None, sfreq=128):
    results = []

    # windows in seconds
    windows_sec = [
        (-3.0, -2.5),
        (-2.5, -2.0),
        (-2.0, -1.5),
        (-1.5, -1.0),
        (-1.0, -0.5),
        (-0.5, 0.0),
    ]

    for subj in np.unique(subject_ids):
        train_mask = (subject_ids != subj)
        test_mask  = (subject_ids == subj)
        
        X_train = X[train_mask]
        X_test = X[test_mask]
        
        y_train = y[train_mask]
        y_test = y[test_mask]
        
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(f"Skipping Subject {subj}: Not enough classes.")
            continue

        try:
            train_features = []
            test_features  = []

            # ==========================================
            # MULTI-WINDOW FEATURE EXTRACTION
            # ==========================================
            for t0, t1 in windows_sec:
                s0 = int((t0 + 3.0) * sfreq)
                s1 = int((t1 + 3.0) * sfreq)

                Xtr_w = X_train[:, :, s0:s1]
                Xte_w = X_test[:, :, s0:s1]

                # covariance
                cov = Covariances(estimator='oas')
                Xtr_cov = cov.fit_transform(Xtr_w)
                Xte_cov = cov.transform(Xte_w)

                # tangent space
                ts = TangentSpace()
                Xtr_ts = ts.fit_transform(Xtr_cov)
                Xte_ts = ts.transform(Xte_cov)

                train_features.append(Xtr_ts)
                test_features.append(Xte_ts)

            # concantenate features
            Xtr_final = np.concatenate(train_features, axis=1)
            Xte_final = np.concatenate(test_features, axis=1)

            # pca = PCA(n_components=0.95)
            # Xtr_final = pca.fit_transform(Xtr_final)
            # Xte_final = pca.transform(Xte_final)

            # ==========================================
            # CLASSIFIER
            # ==========================================
            clf = LogisticRegression(
                max_iter=4000,
                class_weight='balanced',
                C=0.1,
                solver='liblinear'
            )

            clf.fit(Xtr_final, y_train)

            preds = clf.predict(Xte_final)
            probs = clf.predict_proba(Xte_final)[:, 1]

            acc = balanced_accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds, average='macro')
            
            try:
                auc = roc_auc_score(y_test, probs)
            except ValueError:
                auc = np.nan

            
            results.append({
                'subject': subj,
                'acc': acc,
                'f1':  f1,
                'auc': auc
            })

            print(f"Subject {subj} complete -> Acc: {acc:.3f}")

     
        except Exception as e:
            print(f"Error processing subject {subj}: {e}")

    # ==========================================
    # FINAL DATAFRAME & GRAND MEANS
    # ==========================================
    final_df = pd.DataFrame(results)

    if final_df.empty:
        print("No results to display. Pipeline failed for all subjects.")
        return final_df

    print("\n=======================================================================")
    print("OVERALL METRICS PER SUBJECT")
    print("=======================================================================")
    print(final_df.set_index('subject').round(3))

    print("\n=======================================================================")
    print("GRAND MEANS (Συνολική Απόδοση Συστήματος)")
    print("=======================================================================")
    print(f"Balanced Accuracy: {final_df['acc'].mean():.3f} ± {final_df['acc'].std():.3f}")
    print(f"F1-Score (Macro):  {final_df['f1'].mean():.3f} ± {final_df['f1'].std():.3f}")
    print(f"ROC-AUC:           {final_df['auc'].mean():.3f} ± {final_df['auc'].std():.3f}")
    print("=======================================================================")
    
    return final_df

### EEGNET

In [ ]:
import numpy as np
import tensorflow as tf

from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
from EEGModels import EEGNet
import random
import os

SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

tf.config.experimental.enable_op_determinism()

def eegnet_cv_cross(X, y, subject_ids, session_ids):

    results = []

    Chans = X.shape[1]
    Samples = X.shape[2]

    X_reshaped = X[..., np.newaxis]

    fold_results = []
    all_preds = np.full(len(y), -1)
    all_trues = np.full(len(y), -1)

    fold_counter = 0
    
    
    unique_subjects = np.unique(subject_ids)

    for subj in unique_subjects:
        session_accs = []
        remaining = [s for s in unique_subjects if s != subj]
        val_sub = remaining[0]
        train_subs = remaining[1:]
        train_mask = np.isin(subject_ids, train_subs)
        val_mask   = np.isin(subject_ids, val_sub)
        test_mask  = np.isin(subject_ids, [subj])


        

        X_train = X_reshaped[train_mask]
        X_test = X_reshaped[test_mask]
        X_val = X_reshaped[val_mask]

        y_train = y[train_mask]
        y_test = y[test_mask]
        y_val = y[val_mask]

        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            continue

            # Normalization

        train_mean = X_train.mean(axis=2, keepdims=True)
        train_std = X_train.std(axis=2, keepdims=True) + 1e-8
        X_train_norm = (X_train - train_mean) / train_std
        
        # Test and Val must be normalized by their own characteristics if using per-trial 
        X_test_norm = (X_test - X_test.mean(axis=2, keepdims=True)) / (X_test.std(axis=2, keepdims=True) + 1e-8)
        X_val_norm = (X_val - X_val.mean(axis=2, keepdims=True)) / (X_val.std(axis=2, keepdims=True) + 1e-8)
       
        

            # Class weights
        classes = np.unique(y_train)

        weights = compute_class_weight(
            class_weight='balanced',
            classes=classes,
            y=y_train
        )

        class_weight_dict = dict(zip(classes, weights))

        try:

            tf.keras.backend.clear_session()

            model = EEGNet(
                nb_classes=2,
                Chans=Chans,
                Samples=Samples,
                dropoutRate=0.25,
                kernLength=64,
                F1=8,
                D=2,
                F2=16,
                dropoutType='Dropout'
            )

            model.compile(
                loss='sparse_categorical_crossentropy',
                optimizer='adam',
                metrics=['accuracy']
            )

            early_stop = EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True
            )

            model.fit(
                X_train_norm,
                y_train,
                batch_size=16,
                epochs=50,
                verbose=0,
                shuffle=False,
                validation_data=(X_val_norm, y_val),
                callbacks=[early_stop],
                class_weight=class_weight_dict
            )

            probs = model.predict(X_test_norm, verbose=0)
            preds = probs.argmax(axis=-1)

            bal_acc = balanced_accuracy_score(y_test, preds)
            acc = accuracy_score(y_test, preds)

            precision = precision_score(y_test, preds)
            recall = recall_score(y_test, preds)
            f1 = f1_score(y_test, preds)

            roc_auc = roc_auc_score(y_test, probs[:, 1])
                
            
            cm_local = confusion_matrix(y_test, preds)

            if cm_local.shape == (2, 2):
                tn, fp, fn, tp = cm_local.ravel()
                specificity = tn / (tn + fp + 1e-8)
            else:
                specificity = np.nan

            
            session_accs.append(bal_acc)

            fold_counter += 1

            print(
                f"\n── Fold {fold_counter} | "
                f"Subj {subj} | "
                )

            print(f"Balanced Accuracy: {bal_acc:.3f}")

            global_idx = np.where(test_mask)[0]

            all_preds[global_idx] = preds
            all_trues[global_idx] = y_test

            fold_results.append({
                'fold': fold_counter,
                'subject': subj,
                    
                'accuracy': acc,
                'bal_acc': bal_acc,
                'precision': precision,
                'recall': recall,
                'specificity': specificity,
                'f1': f1,
                'roc_auc': roc_auc
                   
            })
            print(f"Accuracy          : {acc:.3f}")
            print(f"Balanced Accuracy : {bal_acc:.3f}")
            print(f"Precision         : {precision:.3f}")
            print(f"Recall/Sensitivity: {recall:.3f}")
            print(f"Specificity       : {specificity:.3f}")
            print(f"F1 Score          : {f1:.3f}")
            print(f"ROC-AUC           : {roc_auc:.3f}")

        except Exception as e:

            print(f"Error in Subject {subj}: {e}")
            continue

        if session_accs:

            subj_mean = np.mean(session_accs)
            subj_std = np.std(session_accs)

            results.append({
                'subject': subj,
                'mean_acc': subj_mean,
                'std_acc': subj_std
            })


    # Final summary
    if fold_results:

        bal_accs = [r['bal_acc'] for r in fold_results]
        accs = [r['accuracy'] for r in fold_results]
        f1_scores = [r['f1'] for r in fold_results]
        auc_roc = [r['roc_auc'] for r in fold_results]
        print(f"\n{'='*65}")
        print(f"Total Valid Splits: {len(fold_results)}")
        print(f"Balanced Accuracy : {np.mean(bal_accs):.3f} ± {np.std(bal_accs):.3f}")
        print(f"Accuracy          : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
        print(f"F1 Score : {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
        print(f"AUC ROC : {np.mean(auc_roc):.3f} ± {np.std(auc_roc):.3f}")
        

    # Confusion matrix
    valid_mask = (all_trues != -1)

    cm = confusion_matrix(
        all_trues[valid_mask],
        all_preds[valid_mask]
    )
    

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(
        all_trues[valid_mask],
        all_preds[valid_mask]
    ))

    return results, fold_results, all_preds, all_trues

